# V8: Policy-Aware Model and Tool Routing

V1–V7 focus primarily on whether the agent can complete a task reliably. V8 adds a complementary systems concern: **which capabilities should be used for each task under quality, cost, latency, and risk constraints?**

The router does not ask an LLM to make an unconstrained infrastructure decision. It follows a hybrid design:

1. deterministic policy creates the allowed action set;
2. capability profiles identify feasible models and tools;
3. the least-cost feasible route is selected;
4. execution can fall back to a stronger model after failure;
5. high-risk or approval-required actions stop for a human.

All price and latency numbers in this notebook are **simulated units**, not vendor prices or benchmark claims.

## 0. Source Map and Design Rationale

- [OpenAI Model Guidance](https://developers.openai.com/api/docs/guides/latest-model): recommends choosing model capability intentionally, exposing only relevant tools, defining concurrency/retry/stopping limits, and comparing quality with tokens, latency, calls, and cost.
- [OpenAI Tool Search](https://developers.openai.com/api/docs/guides/tools-tool-search): supports dynamically loading relevant tool definitions instead of exposing an unnecessarily large tool surface.
- [OpenAI Agent Evals](https://developers.openai.com/api/docs/guides/agent-evals): motivates repeatable task and trajectory evaluation for routing changes.
- [Anthropic: Building Effective Agents](https://www.anthropic.com/engineering/building-effective-agents): describes routing, parallelization, and orchestrator-worker patterns and recommends adding complexity only when measured needs justify it.

V8 treats routing as a constrained optimization problem, not a prompt trick. The model can recommend a route in a future extension, but policy owns authorization and budget enforcement.

## 1. Setup

In [1]:
import json
import os
from enum import Enum

from dotenv import find_dotenv, load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI
from pydantic import BaseModel, Field

load_dotenv(find_dotenv(usecwd=True))

MODEL_NAME = os.getenv("MODEL_NAME", "gpt-5-nano")
USE_LIVE_MODEL = os.getenv("V8_USE_LIVE_MODEL", "1") == "1"
client = OpenAI() if os.getenv("OPENAI_API_KEY") else None

print(f"[V8] Configured live model: {MODEL_NAME}")
print(f"[V8] Live synthesis enabled: {USE_LIVE_MODEL and client is not None}")
print("[V8] Routing costs and latencies are simulated units.")

[V8] Configured live model: gpt-5-nano
[V8] Live synthesis enabled: True
[V8] Routing costs and latencies are simulated units.


## 2. Capability and Routing Contracts

A route is valid only if it satisfies hard constraints before optimization:

- model complexity and risk limits;
- predicted quality floor;
- total cost and latency budgets;
- tool relevance;
- approval policy.

Execution modes are also explicit:

- `direct`: the model must inspect results before its next judgment;
- `parallel`: independent evidence calls can run concurrently;
- `programmatic`: a bounded deterministic transform such as deduplication;
- `human`: autonomous execution is not authorized.

In [2]:
class ExecutionMode(str, Enum):
    DIRECT = "direct"
    PARALLEL = "parallel"
    PROGRAMMATIC = "programmatic"
    HUMAN = "human"


class ModelProfile(BaseModel):
    id: str
    capability_tier: str
    maximum_complexity: int = Field(ge=1, le=5)
    maximum_risk: int = Field(ge=1, le=3)
    predicted_quality: float = Field(ge=0.0, le=1.0)
    cost_units: float = Field(ge=0.0)
    latency_units: int = Field(ge=0)


class ToolProfile(BaseModel):
    name: str
    evidence_lane: str
    cost_units: float = Field(ge=0.0)
    latency_units: int = Field(ge=0)
    side_effecting: bool = False


class RoutingRequest(BaseModel):
    id: str
    description: str
    complexity: int = Field(ge=1, le=5)
    risk: int = Field(ge=1, le=4)
    quality_floor: float = Field(ge=0.0, le=1.0)
    maximum_total_cost: float = Field(gt=0.0)
    maximum_latency: int = Field(gt=0)
    required_evidence_lanes: set[str] = Field(default_factory=set)
    independent_tool_calls: bool = False
    result_dependent_judgment: bool = False
    bounded_transform: bool = False
    requires_human_approval: bool = False


class RoutingDecision(BaseModel):
    request_id: str
    model_id: str | None
    tool_names: list[str]
    execution_mode: ExecutionMode
    fallback_model_ids: list[str]
    predicted_cost: float
    predicted_latency: int
    rationale: list[str]


class ExecutionReceipt(BaseModel):
    request_id: str
    status: str
    selected_model_id: str | None
    attempted_model_ids: list[str]
    tool_names: list[str]
    execution_mode: ExecutionMode
    quality_score: float
    total_cost: float
    total_latency: int
    fallback_count: int

## 3. Simulated Capability Registry

The labels `economy`, `balanced`, and `frontier` are architectural tiers. They deliberately do not claim that a specific vendor model always has these exact properties.

Tool profiles also expose only a narrow evidence lane. This lets the router prove that an academic task receives academic tools while a bounded deduplication task receives a deterministic transform tool.

In [3]:
MODEL_PROFILES = {
    profile.id: profile
    for profile in [
        ModelProfile(
            id="economy",
            capability_tier="efficient",
            maximum_complexity=2,
            maximum_risk=1,
            predicted_quality=0.86,
            cost_units=0.20,
            latency_units=220,
        ),
        ModelProfile(
            id="balanced",
            capability_tier="general",
            maximum_complexity=4,
            maximum_risk=2,
            predicted_quality=0.94,
            cost_units=0.70,
            latency_units=560,
        ),
        ModelProfile(
            id="frontier",
            capability_tier="highest",
            maximum_complexity=5,
            maximum_risk=3,
            predicted_quality=0.99,
            cost_units=2.00,
            latency_units=1_120,
        ),
    ]
}

TOOL_PROFILES = {
    profile.name: profile
    for profile in [
        ToolProfile(
            name="arxiv_search",
            evidence_lane="academic",
            cost_units=0.10,
            latency_units=300,
        ),
        ToolProfile(
            name="official_docs_search",
            evidence_lane="official",
            cost_units=0.05,
            latency_units=180,
        ),
        ToolProfile(
            name="fresh_web_search",
            evidence_lane="fresh_news",
            cost_units=0.12,
            latency_units=260,
        ),
        ToolProfile(
            name="deduplicate_and_rank",
            evidence_lane="structured_transform",
            cost_units=0.03,
            latency_units=80,
        ),
        ToolProfile(
            name="publish_report",
            evidence_lane="external_write",
            cost_units=0.02,
            latency_units=100,
            side_effecting=True,
        ),
    ]
}

print("[V8][REGISTRY] Models:", list(MODEL_PROFILES))
print("[V8][REGISTRY] Tools:", list(TOOL_PROFILES))

[V8][REGISTRY] Models: ['economy', 'balanced', 'frontier']
[V8][REGISTRY] Tools: ['arxiv_search', 'official_docs_search', 'fresh_web_search', 'deduplicate_and_rank', 'publish_report']


## 4. Policy-Aware Router

The router applies hard rules in this order:

1. approval-required or unsupported risk routes to a human;
2. select only tools matching required evidence lanes;
3. select models meeting complexity, risk, quality, cost, and latency constraints;
4. choose the least-cost feasible primary model;
5. build a stronger fallback chain that still fits the total budget;
6. choose the execution mode from task semantics.

This order matters. Optimizing cost before applying safety constraints would produce an unsafe router.

In [4]:
class PolicyRouter:
    def __init__(
        self,
        models: dict[str, ModelProfile],
        tools: dict[str, ToolProfile],
    ):
        self.models = models
        self.tools = tools

    def _select_tools(self, request: RoutingRequest) -> list[ToolProfile]:
        selected = [
            tool
            for tool in self.tools.values()
            if tool.evidence_lane in request.required_evidence_lanes
        ]
        return sorted(selected, key=lambda item: item.name)

    @staticmethod
    def _tool_latency(
        tools: list[ToolProfile],
        mode: ExecutionMode,
    ) -> int:
        if not tools:
            return 0
        if mode == ExecutionMode.PARALLEL:
            return max(tool.latency_units for tool in tools)
        return sum(tool.latency_units for tool in tools)

    def route(self, request: RoutingRequest) -> RoutingDecision:
        selected_tools = self._select_tools(request)
        has_side_effect = any(tool.side_effecting for tool in selected_tools)

        if request.requires_human_approval or has_side_effect or request.risk > 3:
            return RoutingDecision(
                request_id=request.id,
                model_id=None,
                tool_names=[tool.name for tool in selected_tools],
                execution_mode=ExecutionMode.HUMAN,
                fallback_model_ids=[],
                predicted_cost=0.0,
                predicted_latency=0,
                rationale=[
                    "Autonomous execution is blocked by approval or risk policy.",
                ],
            )

        if request.bounded_transform:
            mode = ExecutionMode.PROGRAMMATIC
        elif request.independent_tool_calls and len(selected_tools) > 1:
            mode = ExecutionMode.PARALLEL
        else:
            mode = ExecutionMode.DIRECT

        tool_cost = sum(tool.cost_units for tool in selected_tools)
        tool_latency = self._tool_latency(selected_tools, mode)
        feasible_models = [
            model
            for model in self.models.values()
            if model.maximum_complexity >= request.complexity
            and model.maximum_risk >= request.risk
            and model.predicted_quality >= request.quality_floor
            and model.cost_units + tool_cost <= request.maximum_total_cost
            and model.latency_units + tool_latency <= request.maximum_latency
        ]

        if not feasible_models:
            return RoutingDecision(
                request_id=request.id,
                model_id=None,
                tool_names=[tool.name for tool in selected_tools],
                execution_mode=ExecutionMode.HUMAN,
                fallback_model_ids=[],
                predicted_cost=0.0,
                predicted_latency=0,
                rationale=["No autonomous route satisfies all hard constraints."],
            )

        ordered = sorted(
            feasible_models,
            key=lambda item: (item.cost_units, item.latency_units),
        )
        primary = ordered[0]

        fallback_ids: list[str] = []
        running_cost = primary.cost_units + tool_cost
        for candidate in ordered[1:]:
            if running_cost + candidate.cost_units <= request.maximum_total_cost:
                fallback_ids.append(candidate.id)
                running_cost += candidate.cost_units

        return RoutingDecision(
            request_id=request.id,
            model_id=primary.id,
            tool_names=[tool.name for tool in selected_tools],
            execution_mode=mode,
            fallback_model_ids=fallback_ids,
            predicted_cost=round(primary.cost_units + tool_cost, 2),
            predicted_latency=primary.latency_units + tool_latency,
            rationale=[
                "All hard risk, quality, cost, and latency constraints passed.",
                f"Selected the least-cost feasible model tier: {primary.id}.",
                f"Exposed {len(selected_tools)} task-relevant tool(s).",
                f"Execution mode is {mode.value} based on dependency semantics.",
            ],
        )

## 5. Execution with Bounded Fallback

A route decision is not the same as a successful execution. The executor records attempted models, simulated resource use, fallback count, and the actual quality result.

Failed attempts consume budget. The fallback chain is therefore precomputed so a retry cannot silently exceed the request's maximum cost.

In [5]:
class RoutedExecutor:
    def __init__(
        self,
        models: dict[str, ModelProfile],
        tools: dict[str, ToolProfile],
    ):
        self.models = models
        self.tools = tools

    def execute(
        self,
        request: RoutingRequest,
        decision: RoutingDecision,
        failing_models: set[str] | None = None,
    ) -> ExecutionReceipt:
        failing_models = failing_models or set()
        if decision.execution_mode == ExecutionMode.HUMAN:
            return ExecutionReceipt(
                request_id=request.id,
                status="waiting_for_human",
                selected_model_id=None,
                attempted_model_ids=[],
                tool_names=decision.tool_names,
                execution_mode=decision.execution_mode,
                quality_score=0.0,
                total_cost=0.0,
                total_latency=0,
                fallback_count=0,
            )

        tool_profiles = [self.tools[name] for name in decision.tool_names]
        tool_cost = sum(tool.cost_units for tool in tool_profiles)
        if decision.execution_mode == ExecutionMode.PARALLEL and tool_profiles:
            tool_latency = max(tool.latency_units for tool in tool_profiles)
        else:
            tool_latency = sum(tool.latency_units for tool in tool_profiles)

        candidates = [decision.model_id] + decision.fallback_model_ids
        attempted: list[str] = []
        total_model_cost = 0.0
        total_model_latency = 0
        selected_model: ModelProfile | None = None

        for model_id in candidates:
            model = self.models[model_id]
            attempted.append(model_id)
            total_model_cost += model.cost_units
            total_model_latency += model.latency_units
            if model_id not in failing_models:
                selected_model = model
                break

        if selected_model is None:
            status = "failed"
            quality = 0.0
        else:
            quality = selected_model.predicted_quality
            status = "completed" if quality >= request.quality_floor else "quality_failed"

        return ExecutionReceipt(
            request_id=request.id,
            status=status,
            selected_model_id=selected_model.id if selected_model else None,
            attempted_model_ids=attempted,
            tool_names=decision.tool_names,
            execution_mode=decision.execution_mode,
            quality_score=quality,
            total_cost=round(total_model_cost + tool_cost, 2),
            total_latency=total_model_latency + tool_latency,
            fallback_count=max(len(attempted) - 1, 0),
        )

## 6. Representative Routing Workload

The workload covers distinct decisions rather than variations of one prompt:

- simple synthesis from existing evidence;
- academic research with independent evidence lanes;
- complex result-dependent synthesis;
- a bounded deterministic transform;
- adaptive search requiring fresh judgment;
- an external write requiring approval;
- a transient model failure requiring fallback.

In [6]:
requests = [
    RoutingRequest(
        id="simple_summary",
        description="Summarize evidence already present in context.",
        complexity=1,
        risk=1,
        quality_floor=0.84,
        maximum_total_cost=0.50,
        maximum_latency=400,
    ),
    RoutingRequest(
        id="academic_research",
        description="Collect academic and official evidence on agent memory.",
        complexity=3,
        risk=1,
        quality_floor=0.92,
        maximum_total_cost=3.00,
        maximum_latency=1_500,
        required_evidence_lanes={"academic", "official"},
        independent_tool_calls=True,
    ),
    RoutingRequest(
        id="complex_synthesis",
        description="Resolve conflicting evidence and synthesize a high-stakes architecture review.",
        complexity=5,
        risk=2,
        quality_floor=0.98,
        maximum_total_cost=3.00,
        maximum_latency=1_800,
        required_evidence_lanes={"academic", "official"},
        result_dependent_judgment=True,
    ),
    RoutingRequest(
        id="bounded_deduplication",
        description="Deduplicate and rank a known list of source records.",
        complexity=2,
        risk=1,
        quality_floor=0.84,
        maximum_total_cost=0.50,
        maximum_latency=500,
        required_evidence_lanes={"structured_transform"},
        bounded_transform=True,
    ),
    RoutingRequest(
        id="adaptive_search",
        description="Search, inspect the result, and decide the next evidence query.",
        complexity=3,
        risk=1,
        quality_floor=0.92,
        maximum_total_cost=3.00,
        maximum_latency=1_500,
        required_evidence_lanes={"academic", "official"},
        result_dependent_judgment=True,
    ),
    RoutingRequest(
        id="publish_external",
        description="Publish the final report to an external destination.",
        complexity=2,
        risk=4,
        quality_floor=0.90,
        maximum_total_cost=2.00,
        maximum_latency=1_000,
        required_evidence_lanes={"external_write"},
        requires_human_approval=True,
    ),
    RoutingRequest(
        id="fallback_case",
        description="Research an agent reliability topic while the balanced tier is unavailable.",
        complexity=3,
        risk=1,
        quality_floor=0.92,
        maximum_total_cost=4.00,
        maximum_latency=2_500,
        required_evidence_lanes={"academic", "official"},
        independent_tool_calls=True,
    ),
]

router = PolicyRouter(MODEL_PROFILES, TOOL_PROFILES)
executor = RoutedExecutor(MODEL_PROFILES, TOOL_PROFILES)
decisions = {request.id: router.route(request) for request in requests}

for request in requests:
    decision = decisions[request.id]
    print(
        f"[V8][ROUTE] {request.id}: model={decision.model_id}, "
        f"mode={decision.execution_mode.value}, tools={decision.tool_names}, "
        f"fallbacks={decision.fallback_model_ids}"
    )

[V8][ROUTE] simple_summary: model=economy, mode=direct, tools=[], fallbacks=[]
[V8][ROUTE] academic_research: model=balanced, mode=parallel, tools=['arxiv_search', 'official_docs_search'], fallbacks=['frontier']
[V8][ROUTE] complex_synthesis: model=frontier, mode=direct, tools=['arxiv_search', 'official_docs_search'], fallbacks=[]
[V8][ROUTE] bounded_deduplication: model=economy, mode=programmatic, tools=['deduplicate_and_rank'], fallbacks=[]
[V8][ROUTE] adaptive_search: model=balanced, mode=direct, tools=['arxiv_search', 'official_docs_search'], fallbacks=[]
[V8][ROUTE] publish_external: model=None, mode=human, tools=['publish_report'], fallbacks=[]
[V8][ROUTE] fallback_case: model=balanced, mode=parallel, tools=['arxiv_search', 'official_docs_search'], fallbacks=['frontier']


## 7. Execute Routes and Inject a Model Failure

In [7]:
receipts: dict[str, ExecutionReceipt] = {}
for request in requests:
    failing = {"balanced"} if request.id == "fallback_case" else set()
    receipts[request.id] = executor.execute(
        request,
        decisions[request.id],
        failing_models=failing,
    )

for request_id, receipt in receipts.items():
    print(
        f"[V8][EXECUTE] {request_id}: status={receipt.status}, "
        f"model={receipt.selected_model_id}, cost={receipt.total_cost:.2f}, "
        f"latency={receipt.total_latency}, fallbacks={receipt.fallback_count}"
    )

[V8][EXECUTE] simple_summary: status=completed, model=economy, cost=0.20, latency=220, fallbacks=0
[V8][EXECUTE] academic_research: status=completed, model=balanced, cost=0.85, latency=860, fallbacks=0
[V8][EXECUTE] complex_synthesis: status=completed, model=frontier, cost=2.15, latency=1600, fallbacks=0
[V8][EXECUTE] bounded_deduplication: status=completed, model=economy, cost=0.23, latency=300, fallbacks=0
[V8][EXECUTE] adaptive_search: status=completed, model=balanced, cost=0.85, latency=1040, fallbacks=0
[V8][EXECUTE] publish_external: status=waiting_for_human, model=None, cost=0.00, latency=0, fallbacks=0
[V8][EXECUTE] fallback_case: status=completed, model=frontier, cost=2.85, latency=1980, fallbacks=1


## 8. Policy Router vs Static Frontier Routing

The baseline sends every autonomous request to the frontier tier. The policy route instead uses the least-cost tier that still meets the task's quality floor.

We compare **pass rate first**. Lower cost is an improvement only if the routed system preserves the same task success rate. This prevents a misleading optimization that simply downgrades quality.

In [8]:
autonomous_requests = [
    request for request in requests if request.id != "publish_external"
]
policy_receipts = [receipts[request.id] for request in autonomous_requests]

def static_frontier_receipt(request: RoutingRequest) -> ExecutionReceipt:
    tools = [
        tool
        for tool in TOOL_PROFILES.values()
        if tool.evidence_lane in request.required_evidence_lanes
    ]
    tool_cost = sum(tool.cost_units for tool in tools)
    tool_latency = sum(tool.latency_units for tool in tools)
    frontier = MODEL_PROFILES["frontier"]
    return ExecutionReceipt(
        request_id=request.id,
        status="completed",
        selected_model_id="frontier",
        attempted_model_ids=["frontier"],
        tool_names=[tool.name for tool in tools],
        execution_mode=ExecutionMode.DIRECT,
        quality_score=frontier.predicted_quality,
        total_cost=round(frontier.cost_units + tool_cost, 2),
        total_latency=frontier.latency_units + tool_latency,
        fallback_count=0,
    )

static_receipts = [
    static_frontier_receipt(request) for request in autonomous_requests
]

def summarize(receipt_set: list[ExecutionReceipt]) -> dict[str, float]:
    return {
        "pass_rate": round(
            sum(receipt.status == "completed" for receipt in receipt_set)
            / len(receipt_set),
            3,
        ),
        "total_cost": round(sum(receipt.total_cost for receipt in receipt_set), 2),
        "total_latency": float(sum(receipt.total_latency for receipt in receipt_set)),
        "mean_quality": round(
            sum(receipt.quality_score for receipt in receipt_set)
            / len(receipt_set),
            3,
        ),
    }

policy_summary = summarize(policy_receipts)
static_summary = summarize(static_receipts)

print("[V8][A/B] Policy router:", policy_summary)
print("[V8][A/B] Static frontier:", static_summary)
print(
    "[V8][A/B] Cost reduction:",
    round(1 - policy_summary["total_cost"] / static_summary["total_cost"], 3),
)
print(
    "[V8][A/B] Latency reduction:",
    round(1 - policy_summary["total_latency"] / static_summary["total_latency"], 3),
)

[V8][A/B] Policy router: {'pass_rate': 1.0, 'total_cost': 7.13, 'total_latency': 6000.0, 'mean_quality': 0.93}
[V8][A/B] Static frontier: {'pass_rate': 1.0, 'total_cost': 12.63, 'total_latency': 8720.0, 'mean_quality': 0.99}
[V8][A/B] Cost reduction: 0.435
[V8][A/B] Latency reduction: 0.312


## 9. End-to-End Evaluation

In [9]:
request_by_id = {request.id: request for request in requests}

tool_relevance = all(
    TOOL_PROFILES[tool_name].evidence_lane
    in request_by_id[request_id].required_evidence_lanes
    for request_id, decision in decisions.items()
    for tool_name in decision.tool_names
)
autonomous_within_budget = all(
    receipt.total_cost <= request_by_id[receipt.request_id].maximum_total_cost
    and receipt.total_latency <= request_by_id[receipt.request_id].maximum_latency
    for receipt in policy_receipts
)

checks = {
    "simple_task_uses_economy_tier": decisions["simple_summary"].model_id
    == "economy",
    "academic_task_uses_balanced_tier": decisions["academic_research"].model_id
    == "balanced",
    "complex_task_uses_frontier_tier": decisions["complex_synthesis"].model_id
    == "frontier",
    "independent_research_is_parallel": decisions["academic_research"].execution_mode
    == ExecutionMode.PARALLEL,
    "bounded_transform_is_programmatic": decisions["bounded_deduplication"].execution_mode
    == ExecutionMode.PROGRAMMATIC,
    "result_dependent_search_is_direct": decisions["adaptive_search"].execution_mode
    == ExecutionMode.DIRECT,
    "external_write_waits_for_human": receipts["publish_external"].status
    == "waiting_for_human",
    "only_relevant_tools_exposed": tool_relevance,
    "fallback_completed": receipts["fallback_case"].status == "completed",
    "fallback_escalated_to_frontier": receipts["fallback_case"].selected_model_id
    == "frontier",
    "fallback_was_recorded": receipts["fallback_case"].fallback_count == 1,
    "all_autonomous_routes_within_budget": autonomous_within_budget,
    "policy_preserves_pass_rate": policy_summary["pass_rate"]
    == static_summary["pass_rate"],
    "policy_reduces_cost": policy_summary["total_cost"]
    < static_summary["total_cost"],
    "policy_reduces_latency": policy_summary["total_latency"]
    < static_summary["total_latency"],
    "all_decisions_are_explainable": all(
        decision.rationale for decision in decisions.values()
    ),
}

for name, passed in checks.items():
    print(f"[V8][EVAL] {name}: {'PASS' if passed else 'FAIL'}")

assert all(checks.values()), [name for name, passed in checks.items() if not passed]
print(f"[V8][EVAL] Passed {sum(checks.values())}/{len(checks)} checks.")

[V8][EVAL] simple_task_uses_economy_tier: PASS
[V8][EVAL] academic_task_uses_balanced_tier: PASS
[V8][EVAL] complex_task_uses_frontier_tier: PASS
[V8][EVAL] independent_research_is_parallel: PASS
[V8][EVAL] bounded_transform_is_programmatic: PASS
[V8][EVAL] result_dependent_search_is_direct: PASS
[V8][EVAL] external_write_waits_for_human: PASS
[V8][EVAL] only_relevant_tools_exposed: PASS
[V8][EVAL] fallback_completed: PASS
[V8][EVAL] fallback_escalated_to_frontier: PASS
[V8][EVAL] fallback_was_recorded: PASS
[V8][EVAL] all_autonomous_routes_within_budget: PASS
[V8][EVAL] policy_preserves_pass_rate: PASS
[V8][EVAL] policy_reduces_cost: PASS
[V8][EVAL] policy_reduces_latency: PASS
[V8][EVAL] all_decisions_are_explainable: PASS
[V8][EVAL] Passed 16/16 checks.


## 10. Embedded Routing Report

In [10]:
report_payload = {
    "note": "All costs and latencies are simulated units.",
    "decisions": {
        request_id: decision.model_dump(mode="json")
        for request_id, decision in decisions.items()
    },
    "receipts": {
        request_id: receipt.model_dump(mode="json")
        for request_id, receipt in receipts.items()
    },
    "policy_summary": policy_summary,
    "static_frontier_summary": static_summary,
    "checks": checks,
}

fallback_report = (
    "# V8 Policy-Aware Routing Evaluation\n\n"
    "## Outcome\n\n"
    f"The policy router preserved a {policy_summary['pass_rate']:.0%} autonomous task "
    f"pass rate while reducing simulated total cost from {static_summary['total_cost']:.2f} "
    f"to {policy_summary['total_cost']:.2f} and latency from "
    f"{static_summary['total_latency']:.0f} to {policy_summary['total_latency']:.0f} units.\n\n"
    "## Control Boundaries\n\n"
    "Hard risk, quality, cost, latency, and tool-relevance constraints were applied before "
    "optimization. The external write waited for a human. Independent evidence calls were "
    "parallelized, bounded transformation was programmatic, and result-dependent search "
    "remained direct. A simulated balanced-tier failure escalated through a recorded fallback.\n\n"
    "## Interpretation\n\n"
    "V8 demonstrates capability routing as constrained systems policy. It does not claim "
    "real vendor pricing or universal model quality; production profiles require measured "
    "workload-specific evals and telemetry."
)

live_report_used = False
final_report = fallback_report
if USE_LIVE_MODEL and client is not None:
    try:
        response = client.responses.create(
            model=MODEL_NAME,
            instructions=(
                "Write a concise Markdown engineering report from the V8 routing evaluation JSON. "
                "Clearly state that costs and latencies are simulated. Explain hard constraints, "
                "execution modes, fallback behavior, approval boundaries, and the A/B result. "
                "Output only the report. Do not offer follow-up help."
            ),
            input=json.dumps(report_payload, indent=2),
        )
        if response.output_text.strip():
            final_report = response.output_text.strip()
            live_report_used = True
    except Exception as exc:
        print(f"[V8][REPORT] Live synthesis unavailable; using deterministic report: {type(exc).__name__}")

print(f"[V8][REPORT] Live model used: {live_report_used}")
print(f"[V8][REPORT] Characters: {len(final_report)}")
display(Markdown(final_report))

[V8][REPORT] Live model used: True
[V8][REPORT] Characters: 5484


# V8 Routing Evaluation — Engineering Report (Simulated Costs/Latencies)

Note: All costs and latencies reported here are simulated units.

Executive Summary
- Overall: All decisions completed with a 100% pass rate and high-quality outcomes.
- A/B result: Dynamic routing across models (economy, balanced, frontier) achieves substantially lower aggregate cost and latency than the static Frontier baseline, with a modest decrease in mean quality.
  - Dynamic policy (policy_summary): total_cost 7.13, total_latency 6000.0, mean_quality 0.93, pass_rate 1.0.
  - Static Frontier baseline (static_frontier_summary): total_cost 12.63, total_latency 8720.0, mean_quality 0.99, pass_rate 1.0.
  - Interpretation: Cost and latency improvements are achieved at a small trade-off in average quality.

Hard Constraints, Execution Modes, and Approval Boundaries
- All hard constraints (risk, quality, cost, latency) were satisfied for each decision.
- Approval-boundary exception: publish_external remains blocked by approval/risk policy and is set to human execution.
- Execution modes observed:
  - direct: simple_summary, adaptive_search, complex_synthesis (where possible)
  - parallel: academic_research, fallback_case
  - programmatic: bounded_deduplication
  - human: publish_external
- Tools and exposure: decisions expose the minimum set of relevant tools per task; no extraneous tool exposure across tasks.

Fallback Behavior and Resilience
- Fallbacks enabled and recorded where applicable:
  - fallback_case demonstrates a fallback path from balanced to frontier (parallel) with 1 fallback event; total_cost 2.85, total_latency 1980.
- Fallback status:
  - Fallback completed: true
  - Fallback escalated to frontier: true
  - Fallback was recorded: true
- External write: publish_external is awaiting human approval, effectively blocking autonomous completion in that branch.

A/B Result and Baseline Comparison
- Baseline: Static frontier route shows higher cost and latency but slightly higher mean quality.
  - Frontier baseline: total_cost 12.63, total_latency 8720.0, mean_quality 0.99, pass_rate 1.0.
- Dynamic routing performance (policy-driven mix of models and modes) reduces cost and latency significantly while maintaining high pass rate (1.0) and acceptable mean quality (0.93).
- Practical takeaway: The adaptive/multi-model routing improves efficiency (cost/latency) with a small quality trade-off compared to a static frontier approach.

Decision Snapshot (Selected Highlights)
- simple_summary
  - model_id: economy
  - execution_mode: direct
  - predicted_cost/latency: 0.2 / 220
  - rationale: passed constraints; least-cost feasible; no tools exposed; direct execution.
  - receipts: completed; quality 0.86; total_cost 0.2; total_latency 220.
- academic_research
  - model_id: balanced
  - execution_mode: parallel
  - tools: arxiv_search, official_docs_search
  - predicted_cost/latency: 0.85 / 860
  - receipts: completed; quality 0.94; total_cost 0.85; total_latency 860.
- complex_synthesis
  - model_id: frontier
  - execution_mode: direct
  - tools: arxiv_search, official_docs_search
  - predicted_cost/latency: 2.15 / 1600
  - receipts: completed; quality 0.99; total_cost 2.15; total_latency 1600.
- bounded_deduplication
  - model_id: economy
  - execution_mode: programmatic
  - tool: deduplicate_and_rank
  - predicted_cost/latency: 0.23 / 300
  - receipts: completed; quality 0.86; total_cost 0.23; total_latency 300.
- adaptive_search
  - model_id: balanced
  - execution_mode: direct
  - tools: arxiv_search, official_docs_search
  - predicted_cost/latency: 0.85 / 1040
  - receipts: completed; quality 0.94; total_cost 0.85; total_latency 1040.
- publish_external
  - model_id: null
  - execution_mode: human
  - tool: publish_report
  - notes: autonomous execution blocked by policy
  - receipts: waiting_for_human; quality 0.0; total_cost 0.0; total_latency 0.
- fallback_case
  - initial: balanced, then frontier (parallel)
  - total_cost/latency: 2.85 / 1980
  - receipts: completed; quality 0.99; fallback_count 1; tools: arxiv_search, official_docs_search.

Quality, Cost, and Latency Metrics
- Policy-level aggregates:
  - pass_rate: 1.0
  - total_cost: 7.13
  - total_latency: 6000.0
  - mean_quality: 0.93
- Static baseline:
  - pass_rate: 1.0
  - total_cost: 12.63
  - total_latency: 8720.0
  - mean_quality: 0.99
- Observations:
  - Dynamic routing achieves notable reductions in cost and latency.
  - Slight reduction in mean_quality compared to the static frontier baseline, within acceptable bounds given policy observability.
  - All autonomous routes remain within budget and are explainable per checks.

Compliance and Observability Highlights
- All checks pass: e.g., independent_research_is_parallel, result_dependent_search_is_direct, external_write_waits_for_human, fallback_was_recorded, policy_preserves_pass_rate, policy_reduces_cost, policy_reduces_latency, and all decisions are explainable.
- Execution trace shows deliberate use of appropriate modes per task, with clear fallback and human-in-the-loop boundary for external publishing.

Conclusion
- The V8 routing evaluation demonstrates effective, simulated-cost routing across multiple model tiers and modes, achieving substantial efficiency gains over a static frontier baseline while maintaining high decision explainability and stable pass rates. A controlled human-in-the-loop boundary is reserved for external publishing tasks, aligning with approval policies.

## 11. V8 Completion Summary

V8 adds an optimization and governance layer over the earlier agent runtime:

`request -> hard constraints -> relevant capability set -> least-cost feasible route -> bounded execution -> fallback or approval -> receipt -> A/B evaluation`

With V8, the educational series covers not only how an agent reasons and acts, but also how a harness governs learning, memory, capability selection, safety boundaries, and resource tradeoffs.